In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
# import trees, usda info, usda names
c_usda = pd.read_csv("../data/raw/usda_characteristics.csv")
n_usda = pd.read_csv("../data/raw/plantlst(1).txt")
trees = gpd.read_file("../data/processed/philly_trees.geojson")


In [3]:
c_usda

,plant_code,Morphology/Physiology | Active Growth Period,Morphology/Physiology | Fall Conspicuous,Morphology/Physiology | Flower Color,Morphology/Physiology | Flower Conspicuous,Morphology/Physiology | Foliage Color,Morphology/Physiology | Foliage Porosity Summer,Morphology/Physiology | Foliage Porosity Winter,Morphology/Physiology | Fruit/Seed Color,Morphology/Physiology | Fruit/Seed Conspicuous,Morphology/Physiology | Leaf Retention,Reproduction | Bloom Period,Reproduction | Fruit/Seed Abundance,Reproduction | Fruit/Seed Period Begin,Reproduction | Fruit/Seed Period End,Suitability/Use | Palatable Human
0,ABBA,Spring and Summer,No,Yellow,No,Green,Dense,Dense,Brown,No,Yes,Mid Summer,Medium,Fall,Fall,No
1,ABFR,Spring and Summer,No,Purple,No,Dark Green,Moderate,Moderate,Brown,Yes,Yes,Mid Spring,Medium,Spring,Fall,No
2,ACGI,Spring and Summer,Yes,White,No,Green,Dense,Moderate,Brown,Yes,No,Mid Spring,High,Summer,Fall,No
3,ACNE2,Spring and Summer,Yes,White,No,Green,Dense,Porous,Brown,Yes,No,Early Spring,High,Summer,Fall,Yes
4,ACNI5,Spring and Summer,Yes,Yellow,No,Green,Dense,Porous,Brown,No,No,Late Spring,Medium,Summer,Fall,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,ULPA,Spring and Summer,Yes,Yellow,No,Green,Dense,Porous,Brown,Yes,No,Late Summer,High,Spring,Spring,No
156,ULPU,Spring and Summer,No,Green,No,Green,Dense,Porous,Brown,No,No,Mid Spring,High,Spring,Spring,No
157,ULRU,Spring and Summer,No,Yellow,No,Green,Moderate,Porous,Brown,No,No,Late Winter,High,Spring,Spring,No
158,ULTH,Spring and Summer,No,Yellow,No,Green,Dense,Porous,Brown,No,No,Mid Spring,Medium,Spring,Spring,No


In [4]:
# Dropping plant code duplicates by using common name as a proxy
n_usda = n_usda.dropna(subset="Common Name")
# Clean up work by officially dropping plant code duplicates
n_usda = n_usda.drop_duplicates(subset="Symbol", ignore_index=True)
# Prepare usda names file to merge with usda data. Need scientific names
n_usda["Scientific Name with Author"] = n_usda["Scientific Name with Author"].str.lower()
n_usda["scientific_name"] = (
    n_usda["Scientific Name with Author"]
.str.split(n=2)
.str[:2]
.str.join(" ")
)



In [5]:
def remove_x(string):
    if " × " in string:
        pieces = string.partition(" ×")
        return pieces[0] + pieces[2]
    elif "×" in string:
        pieces = string.partition("×")
        return pieces[0] + pieces[2]
    elif " x " in string:
        pieces = string.partition(" x")
        return pieces[0] + pieces[2]
    else:
        return string

In [6]:
# used this cell to test my x removing function
test = "Acer × fremani"
print(remove_x(test))

Acer fremani


In [7]:
# remove x from scientific name column
n_usda["scientific_name_c"] = pd.Series([remove_x(n_usda["scientific_name"][i]) for i in range(len(n_usda))])

In [8]:
# drop old scientific name column and rename new one
n_usda = n_usda.drop(columns="scientific_name")
n_usda = n_usda.rename(columns={"scientific_name_c": "scientific_name", "Symbol": "plant_code", "Common Name": "common_name"})

In [9]:
# drop unused columns from n_usda to prepare for merge
n_usda = n_usda.drop(columns=["Synonym Symbol", "Scientific Name with Author", "Family"])
n_usda

,plant_code,common_name,scientific_name
0,ABAB,shrubby Indian mallow,abutilon abutiloides
1,ABAB70,abietinella moss,abietinella abietina
2,ABAL,Ramshaw Meadows sand verbena,abronia alpina
3,ABAL3,silver fir,abies alba
4,ABAM,Pacific silver fir,abies amabilis
...,...,...,...
43771,ZYRE,Reinwardt's zygodon moss,zygodon reinwardtii
43772,ZYSE,octopus fern,zygophlebia sectifrons
43773,ZYVI2,zygodon moss,zygodon viridissimus
43774,ZYVIR,zygodon moss,zygodon viridissimus


In [10]:
# merge usda names with characteristics
m_usda = pd.merge(c_usda, n_usda, on="plant_code", how="left")

In [11]:

# manually adding misses from USDA databse
# Change london planetree from platanus hispanica to platanus acerifolia
m_usda.loc[m_usda['plant_code'] == 'PLHI', 'scientific_name'] = 'platanus acerifolia'
# change bald cyprus to match philly conventions
m_usda.loc[m_usda['plant_code'] == 'TADI2', 'scientific_name'] = 'taxodium distichum'
m_usda.loc[m_usda['plant_code'] == 'TADI2', 'common_name'] = 'baldcypress'

# Adding thornless honeylocust
if 'GLTRI' not in m_usda['plant_code'].values:
    m_usda = pd.concat([m_usda, pd.DataFrame([{
        'plant_code': 'GLTRI',
        'Morphology/Physiology | Active Growth Period': 'Spring and Summer',
        'Morphology/Physiology | Fall Conspicuous': 'Yes',
        'Morphology/Physiology | Flower Color': 'Green',
        'Morphology/Physiology | Flower Conspicuous': 'No',
        'Morphology/Physiology | Foliage Color': 'Green',
        'Morphology/Physiology | Foliage Porosity Summer': 'Moderate',
        'Morphology/Physiology | Foliage Porosity Winter': 'Porous',
        'Morphology/Physiology | Fruit/Seed Color': 'Brown',
        'Morphology/Physiology | Fruit/Seed Conspicuous': 'Yes',
        'Morphology/Physiology | Leaf Retention': 'No',
        'Reproduction | Bloom Period': 'Late Spring',
        'Reproduction | Fruit/Seed Abundance': 'Medium',
        'Reproduction | Fruit/Seed Period Begin': 'Summer',
        'Reproduction | Fruit/Seed Period End': 'Fall',
        'Suitability/Use | Palatable Human': 'No',
        'common_name': 'thornless honeylocust',
        'scientific_name': 'gleditsia triacanthos inermis'
    }])], ignore_index=True)
m_usda


,plant_code,Morphology/Physiology | Active Growth Period,Morphology/Physiology | Fall Conspicuous,Morphology/Physiology | Flower Color,Morphology/Physiology | Flower Conspicuous,Morphology/Physiology | Foliage Color,Morphology/Physiology | Foliage Porosity Summer,Morphology/Physiology | Foliage Porosity Winter,Morphology/Physiology | Fruit/Seed Color,Morphology/Physiology | Fruit/Seed Conspicuous,Morphology/Physiology | Leaf Retention,Reproduction | Bloom Period,Reproduction | Fruit/Seed Abundance,Reproduction | Fruit/Seed Period Begin,Reproduction | Fruit/Seed Period End,Suitability/Use | Palatable Human,common_name,scientific_name
0,ABBA,Spring and Summer,No,Yellow,No,Green,Dense,Dense,Brown,No,Yes,Mid Summer,Medium,Fall,Fall,No,balsam fir,abies balsamea
1,ABFR,Spring and Summer,No,Purple,No,Dark Green,Moderate,Moderate,Brown,Yes,Yes,Mid Spring,Medium,Spring,Fall,No,Fraser fir,abies fraseri
2,ACGI,Spring and Summer,Yes,White,No,Green,Dense,Moderate,Brown,Yes,No,Mid Spring,High,Summer,Fall,No,Amur maple,acer ginnala
3,ACNE2,Spring and Summer,Yes,White,No,Green,Dense,Porous,Brown,Yes,No,Early Spring,High,Summer,Fall,Yes,boxelder,acer negundo
4,ACNI5,Spring and Summer,Yes,Yellow,No,Green,Dense,Porous,Brown,No,No,Late Spring,Medium,Summer,Fall,Yes,black maple,acer nigrum
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,ULPU,Spring and Summer,No,Green,No,Green,Dense,Porous,Brown,No,No,Mid Spring,High,Spring,Spring,No,Siberian elm,ulmus pumila
157,ULRU,Spring and Summer,No,Yellow,No,Green,Moderate,Porous,Brown,No,No,Late Winter,High,Spring,Spring,No,slippery elm,ulmus rubra
158,ULTH,Spring and Summer,No,Yellow,No,Green,Dense,Porous,Brown,No,No,Mid Spring,Medium,Spring,Spring,No,rock elm,ulmus thomasii
159,ZESE80,Spring and Summer,Yes,Green,No,Dark Green,Moderate,Porous,Brown,No,No,Mid Spring,High,Summer,Fall,No,Japanese zelkova,zelkova serrata


In [12]:
m_usda.loc[m_usda['plant_code'] == 'ILOP']

,plant_code,Morphology/Physiology | Active Growth Period,Morphology/Physiology | Fall Conspicuous,Morphology/Physiology | Flower Color,Morphology/Physiology | Flower Conspicuous,Morphology/Physiology | Foliage Color,Morphology/Physiology | Foliage Porosity Summer,Morphology/Physiology | Foliage Porosity Winter,Morphology/Physiology | Fruit/Seed Color,Morphology/Physiology | Fruit/Seed Conspicuous,Morphology/Physiology | Leaf Retention,Reproduction | Bloom Period,Reproduction | Fruit/Seed Abundance,Reproduction | Fruit/Seed Period Begin,Reproduction | Fruit/Seed Period End,Suitability/Use | Palatable Human,common_name,scientific_name
67,ILOP,Spring and Summer,Yes,Yellow,No,Green,Dense,Dense,Red,Yes,Yes,Mid Spring,Low,Summer,Fall,No,American holly,ilex opaca


In [13]:
# normalize tree string data
trees["scientific_name"] = trees["scientific_name"].str.lower()
m_usda["common_name"] = m_usda["common_name"].str.lower()
m_usda["scientific_name"] = m_usda["scientific_name"].str.lower()

In [14]:
# Match common names from usda and philly trees
common_usda_phl = pd.merge(trees, m_usda.drop(columns="scientific_name"), on="common_name")
common_usda_phl

,objectid,tree_name,scientific_name,common_name,Genus,Species,geometry,plant_code,Morphology/Physiology | Active Growth Period,Morphology/Physiology | Fall Conspicuous,...,Morphology/Physiology | Foliage Porosity Summer,Morphology/Physiology | Foliage Porosity Winter,Morphology/Physiology | Fruit/Seed Color,Morphology/Physiology | Fruit/Seed Conspicuous,Morphology/Physiology | Leaf Retention,Reproduction | Bloom Period,Reproduction | Fruit/Seed Abundance,Reproduction | Fruit/Seed Period Begin,Reproduction | Fruit/Seed Period End,Suitability/Use | Palatable Human
0,5,Acer pseudoplatanus - sycamore maple,acer pseudoplatanus,sycamore maple,Acer,pseudoplatanus,POINT (-75.21028 39.98376),ACPS,Spring and Summer,No,...,Dense,Porous,Green,Yes,No,Late Spring,High,Spring,Summer,No
1,6,Ilex opaca - american holly,ilex opaca,american holly,Ilex,opaca,POINT (-75.21047 39.98401),ILOP,Spring and Summer,Yes,...,Dense,Dense,Red,Yes,Yes,Mid Spring,Low,Summer,Fall,No
2,8,Koelreuteria paniculata - goldenrain tree,koelreuteria paniculata,goldenrain tree,Koelreuteria,paniculata,POINT (-75.21008 39.98394),KOPA,Spring,Yes,...,Moderate,Porous,Brown,Yes,No,Late Spring,High,Summer,Fall,No
3,11,Metasequoia glyptostroboides - dawn redwood,metasequoia glyptostroboides,dawn redwood,Metasequoia,glyptostroboides,POINT (-75.20942 39.98372),MEGL8,"Spring, Summer, Fall",Yes,...,Dense,Porous,Brown,No,No,Spring,Low,Summer,Summer,No
4,12,Acer rubrum - red maple,acer rubrum,red maple,Acer,rubrum,POINT (-75.05038 40.0622),ACRU,Spring and Summer,Yes,...,Dense,Porous,Red,Yes,No,Early Spring,High,Spring,Spring,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89562,151722,Platanus x acerifolia - london planetree,platanus acerifolia,london planetree,Platanus,acerifolia,POINT (-75.20873 39.94644),PLHI,Spring and Summer,Yes,...,Dense,Porous,Brown,No,No,Late Spring,Medium,Summer,Fall,No
89563,151723,Platanus x acerifolia - london planetree,platanus acerifolia,london planetree,Platanus,acerifolia,POINT (-75.20876 39.9466),PLHI,Spring and Summer,Yes,...,Dense,Porous,Brown,No,No,Late Spring,Medium,Summer,Fall,No
89564,151724,Gleditsia triacanthos - honeylocust,gleditsia triacanthos,honeylocust,Gleditsia,triacanthos,POINT (-75.16226 39.99548),GLTR,Spring and Summer,Yes,...,Dense,Moderate,Green,Yes,No,Late Spring,High,Spring,Fall,No
89565,151725,Gleditsia triacanthos - honeylocust,gleditsia triacanthos,honeylocust,Gleditsia,triacanthos,POINT (-75.16221 39.9955),GLTR,Spring and Summer,Yes,...,Dense,Moderate,Green,Yes,No,Late Spring,High,Spring,Fall,No


In [15]:
# match common names from usda and philly trees
scientific_usda_phl = pd.merge(trees, m_usda.drop(columns="common_name"), on="scientific_name")
scientific_usda_phl

,objectid,tree_name,scientific_name,common_name,Genus,Species,geometry,plant_code,Morphology/Physiology | Active Growth Period,Morphology/Physiology | Fall Conspicuous,...,Morphology/Physiology | Foliage Porosity Summer,Morphology/Physiology | Foliage Porosity Winter,Morphology/Physiology | Fruit/Seed Color,Morphology/Physiology | Fruit/Seed Conspicuous,Morphology/Physiology | Leaf Retention,Reproduction | Bloom Period,Reproduction | Fruit/Seed Abundance,Reproduction | Fruit/Seed Period Begin,Reproduction | Fruit/Seed Period End,Suitability/Use | Palatable Human
0,1,Ginkgo biloba - ginkgo,ginkgo biloba,ginkgo,Ginkgo,biloba,POINT (-75.2105 39.98383),GIBI2,Spring and Summer,Yes,...,Porous,Porous,Yellow,No,No,NaN,NaN,NaN,NaN,No
1,5,Acer pseudoplatanus - sycamore maple,acer pseudoplatanus,sycamore maple,Acer,pseudoplatanus,POINT (-75.21028 39.98376),ACPS,Spring and Summer,No,...,Dense,Porous,Green,Yes,No,Late Spring,High,Spring,Summer,No
2,6,Ilex opaca - american holly,ilex opaca,american holly,Ilex,opaca,POINT (-75.21047 39.98401),ILOP,Spring and Summer,Yes,...,Dense,Dense,Red,Yes,Yes,Mid Spring,Low,Summer,Fall,No
3,8,Koelreuteria paniculata - goldenrain tree,koelreuteria paniculata,goldenrain tree,Koelreuteria,paniculata,POINT (-75.21008 39.98394),KOPA,Spring,Yes,...,Moderate,Porous,Brown,Yes,No,Late Spring,High,Summer,Fall,No
4,11,Metasequoia glyptostroboides - dawn redwood,metasequoia glyptostroboides,dawn redwood,Metasequoia,glyptostroboides,POINT (-75.20942 39.98372),MEGL8,"Spring, Summer, Fall",Yes,...,Dense,Porous,Brown,No,No,Spring,Low,Summer,Summer,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103001,151722,Platanus x acerifolia - london planetree,platanus acerifolia,london planetree,Platanus,acerifolia,POINT (-75.20873 39.94644),PLHI,Spring and Summer,Yes,...,Dense,Porous,Brown,No,No,Late Spring,Medium,Summer,Fall,No
103002,151723,Platanus x acerifolia - london planetree,platanus acerifolia,london planetree,Platanus,acerifolia,POINT (-75.20876 39.9466),PLHI,Spring and Summer,Yes,...,Dense,Porous,Brown,No,No,Late Spring,Medium,Summer,Fall,No
103003,151724,Gleditsia triacanthos - honeylocust,gleditsia triacanthos,honeylocust,Gleditsia,triacanthos,POINT (-75.16226 39.99548),GLTR,Spring and Summer,Yes,...,Dense,Moderate,Green,Yes,No,Late Spring,High,Spring,Fall,No
103004,151725,Gleditsia triacanthos - honeylocust,gleditsia triacanthos,honeylocust,Gleditsia,triacanthos,POINT (-75.16221 39.9955),GLTR,Spring and Summer,Yes,...,Dense,Moderate,Green,Yes,No,Late Spring,High,Spring,Fall,No


In [16]:
# concat both merged lists then drop duplicates in object ID
usda_matched = pd.concat([common_usda_phl, scientific_usda_phl], axis=0)
usda_matched

,objectid,tree_name,scientific_name,common_name,Genus,Species,geometry,plant_code,Morphology/Physiology | Active Growth Period,Morphology/Physiology | Fall Conspicuous,...,Morphology/Physiology | Foliage Porosity Summer,Morphology/Physiology | Foliage Porosity Winter,Morphology/Physiology | Fruit/Seed Color,Morphology/Physiology | Fruit/Seed Conspicuous,Morphology/Physiology | Leaf Retention,Reproduction | Bloom Period,Reproduction | Fruit/Seed Abundance,Reproduction | Fruit/Seed Period Begin,Reproduction | Fruit/Seed Period End,Suitability/Use | Palatable Human
0,5,Acer pseudoplatanus - sycamore maple,acer pseudoplatanus,sycamore maple,Acer,pseudoplatanus,POINT (-75.21028 39.98376),ACPS,Spring and Summer,No,...,Dense,Porous,Green,Yes,No,Late Spring,High,Spring,Summer,No
1,6,Ilex opaca - american holly,ilex opaca,american holly,Ilex,opaca,POINT (-75.21047 39.98401),ILOP,Spring and Summer,Yes,...,Dense,Dense,Red,Yes,Yes,Mid Spring,Low,Summer,Fall,No
2,8,Koelreuteria paniculata - goldenrain tree,koelreuteria paniculata,goldenrain tree,Koelreuteria,paniculata,POINT (-75.21008 39.98394),KOPA,Spring,Yes,...,Moderate,Porous,Brown,Yes,No,Late Spring,High,Summer,Fall,No
3,11,Metasequoia glyptostroboides - dawn redwood,metasequoia glyptostroboides,dawn redwood,Metasequoia,glyptostroboides,POINT (-75.20942 39.98372),MEGL8,"Spring, Summer, Fall",Yes,...,Dense,Porous,Brown,No,No,Spring,Low,Summer,Summer,No
4,12,Acer rubrum - red maple,acer rubrum,red maple,Acer,rubrum,POINT (-75.05038 40.0622),ACRU,Spring and Summer,Yes,...,Dense,Porous,Red,Yes,No,Early Spring,High,Spring,Spring,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103001,151722,Platanus x acerifolia - london planetree,platanus acerifolia,london planetree,Platanus,acerifolia,POINT (-75.20873 39.94644),PLHI,Spring and Summer,Yes,...,Dense,Porous,Brown,No,No,Late Spring,Medium,Summer,Fall,No
103002,151723,Platanus x acerifolia - london planetree,platanus acerifolia,london planetree,Platanus,acerifolia,POINT (-75.20876 39.9466),PLHI,Spring and Summer,Yes,...,Dense,Porous,Brown,No,No,Late Spring,Medium,Summer,Fall,No
103003,151724,Gleditsia triacanthos - honeylocust,gleditsia triacanthos,honeylocust,Gleditsia,triacanthos,POINT (-75.16226 39.99548),GLTR,Spring and Summer,Yes,...,Dense,Moderate,Green,Yes,No,Late Spring,High,Spring,Fall,No
103004,151725,Gleditsia triacanthos - honeylocust,gleditsia triacanthos,honeylocust,Gleditsia,triacanthos,POINT (-75.16221 39.9955),GLTR,Spring and Summer,Yes,...,Dense,Moderate,Green,Yes,No,Late Spring,High,Spring,Fall,No


In [17]:
#drop duplicate trees from my matched jawns
usda_matched = usda_matched.drop_duplicates(subset="objectid", ignore_index=True)

In [18]:
# add matched trees back to original tree dataset and remove duplicates
matched_trees = pd.concat([trees, usda_matched])
#sort by plant_code so that all trees with no code end up at bottom of df
matched_trees = matched_trees.sort_values(by="plant_code")
# drop object id duplicates and keep first one
# should work because all non matched ones will be at bottom of df
matched_trees = matched_trees.drop_duplicates(subset="objectid", keep="first")

In [19]:
# export this clean geojson to replace the old fucked up one
matched_trees.to_file("/Users/prince/philly-tree-mapper/data/processed/usda_philly_trees.geojson", driver="GeoJSON")